# Tari'ak Traffic Analysis, 01 — SQL Traffic Analysis

This notebook uses SQLite to analyze the cleaned Tari'ak traffic observations. The focus is on temporal coverage, observed velocity patterns, and road-segment behavior. All conclusions should describe the recorded observations carefully and should not treat velocity alone as a direct measure of congestion or travel time.

## 1. Load the processed data into SQLite

The cleaned CSV contains more than six million rows, so it is loaded into SQLite in chunks. The resulting local database allows the remaining analysis to use SQL without repeatedly loading the full CSV into a Pandas dataframe.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

BASE_DIR = Path('..')
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_FILE = PROCESSED_DIR / 'traffic_clean.csv'
DB_PATH = PROCESSED_DIR / 'traffic.db'
TABLE_NAME = 'traffic_observations'
CHUNK_SIZE = 250_000

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.3f}'.format)

def run_query(query):
    return pd.read_sql_query(query, conn)

print(f'Processed CSV exists: {PROCESSED_FILE.exists()}')
print(f'Processed CSV size: {PROCESSED_FILE.stat().st_size / 1024**2:,.1f} MB')

Processed CSV exists: True
Processed CSV size: 581.5 MB


In [2]:
REBUILD_DATABASE = True

conn = sqlite3.connect(DB_PATH)

if REBUILD_DATABASE:
    for chunk_number, chunk in enumerate(pd.read_csv(PROCESSED_FILE, chunksize=CHUNK_SIZE), start=1):
        chunk.to_sql(
            TABLE_NAME,
            conn,
            if_exists='replace' if chunk_number == 1 else 'append',
            index=False
        )

        if chunk_number % 10 == 0:
            print(f'Loaded {chunk_number * CHUNK_SIZE:,} rows...')

    conn.execute(f'CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_timestamp ON {TABLE_NAME} (timestamp)')
    conn.execute(f'CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_osm_id ON {TABLE_NAME} (osm_id)')
    conn.execute(f'CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_hour ON {TABLE_NAME} (hour)')
    conn.commit()

print(f'Connected to: {DB_PATH}')
print(f'Active table: {TABLE_NAME}')

Loaded 2,500,000 rows...
Loaded 5,000,000 rows...
Connected to: ..\data\processed\traffic.db
Active table: traffic_observations
